# Cognopolis · Урок M2 — боевой цикл: от гоблина до волка

Ты управляешь жителем игры **Cognopolis** не мышкой, а **кодом-агентом**. В уроке M1 житель мирно собирал ресурсы — теперь в мире завелись **мобы**, а у боя появилась экономика: трофеи продаются за золото, снаряжение покупается, изнашивается и чинится. Голый житель больше **не тянет волка** (шанс победы ~12%) — ключ от волка — **копьё**.

**Что построим.** Реактивного бойца с полной кампанией: **фарми гоблинов → продай трофеи → накопи 30 г → купи копьё в Лавке → надень → иди на волка → следи за прочностью → чини**. Цель агент выбирает не «на глаз», а по официальному **бестиарию** `GET /enemies` — у каждого врага есть готовый вердикт `danger_at_base`: `safe` (по зубам голому) / `gear_required` (нужна вещь) / `wall` (непобедим — **никогда не лезь**, это огр). А перед рискованным боем агент делает **dry-run** `fight/preview` — точный прогноз стычки без затрат и без риска.

**Тир агента:** реактивный+ (майлстоун игры **M2**). Тот же цикл `observe → decide → act → wait`, что в M1, плюс лесенка правил безопасности и экономика снаряжения. **Дальше по курсу:** планировщик (M3), LLM-агент (M4).

> ⚙️ **Working-first.** Ноутбук рассчитан на прогон `Run all` без правок — нужно лишь задать `BASE_URL` живого мира и свой `COGNOPOLIS_TOKEN` (из Ратуши). Учебная активность — в секции **«Задачи»**. Базовый агент уже безопасно ведёт кампанию; твоё дело — сделать его выносливее (hp-guard) и хозяйственнее (ремонт).

**Ссылки** (подставь адрес своего мира вместо `<BASE_URL>`):
- API-доки (Swagger, кнопка **Authorize**): `<BASE_URL>/docs`
- 👀 Смотреть за своим агентом в браузере: `<BASE_URL>/?token=<твой токен>` (read-only)
- Контракт: житель видит мир **только** через API (`cognopolis_client`).
- Раньше этого — пройди **реактивного сборщика (M1)**.

## 1. Сетап

Ставим официальный клиент игры из git-URL (PyPI пока нет) и задаём адрес мира.

In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis.git#subdirectory=client"

In [ ]:
import os
from cognopolis_client import Client, GameError

# ⬇️ ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Можно переопределить переменной COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

# ⬇️ ТВОЙ ТОКЕН ДОСТУПА (полный доступ к твоему жителю).
#   1. Открой BASE_URL в браузере и зарегистрируйся (логин + пароль).
#   2. Ратуша → раздел «ваш аккаунт» → кнопка «копировать» — это твой токен.
#   3. Вставь его в COGNOPOLIS_TOKEN ниже (или задай переменную окружения / секрет Colab/Kaggle).
TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")  # ← вставь токен в кавычки, если не используешь env
assert TOKEN, f"Вставь токен: зарегистрируйся на {BASE_URL}, скопируй токен из Ратуши и задай COGNOPOLIS_TOKEN."

c = Client(BASE_URL, token=TOKEN)

# Мягкая проверка связи: если мир недоступен — не пугаем трейсбеком, живые ячейки ниже пропустятся.
WORLD_UP = True
try:
    Client(BASE_URL).get_map()  # GET /map не требует токена
except Exception as e:
    WORLD_UP = False
    print(f"⚠️  Мир {BASE_URL} недоступен ({type(e).__name__}). Живые ячейки пропущу — проверь COGNOPOLIS_URL или попробуй позже.")

print("Мир:", BASE_URL, "| на связи:", WORLD_UP)
# Тир M2 — реактивный+: LLM не нужен (он появится на M4).

## 2. Разогрев — смотрим на себя и на врагов

У бойца три пары «глаз», и все — через API:

- `get_character()` — **я сам**: hp, золото в общей казне (`gold`) и слот оружия (`equipment.weapon` — вещь, её прочность и флаг `active`);
- `get_enemies()` — **бестиарий** (каталог, без токена): карточки врагов с hp/атакой/бронёй, ценой трофея и готовым вердиктом `danger_at_base` — `safe` / `gear_required` (в `target_build` подсказана нужная вещь) / `wall`;
- `get_map()["enemies"]` — **враги сейчас**: живые инстансы на карте; жив ли враг, решаем по полю `alive`, а не по картинке клетки.

Бестиарий — статичный справочник «кого вообще можно бить», карта — оперативная сводка «кто где стоит». Хороший агент читает оба и ничего не хардкодит.

In [ ]:
if WORLD_UP:
    print("👀 Смотри за жителем в браузере:", f"{BASE_URL}/?token={TOKEN}")

    ch = c.get_character()
    w = ch["equipment"]["weapon"]
    print("позиция:", (ch["x"], ch["y"]), "| hp:", f'{ch["hp"]}/{ch["max_hp"]}',
          "| золото (общая казна):", ch["gold"],
          "| оружие:", f'{w["item"]} {w["durability"]}/{w["durability_max"]}' if w else "нет")

    # Бестиарий: вердикт danger_at_base — safe / gear_required / wall. wall (огр) — НИКОГДА.
    for e in c.get_enemies()["enemies"]:
        need = e["target_build"].get("gear", "")
        print(f'  {e["name_ru"]:7s} hp {e["max_hp"]:3d} | atk {e["atk_min"]}-{e["atk_max"]} | броня {e["armor"]}'
              f' | трофей {e["loot"]} (нетто {e["bounty_net"]} г) | вердикт: {e["danger_at_base"]}'
              + (f" → нужна вещь: {need}" if need else ""))

    # Живые враги сейчас — на карте (инстансы каталога): жив ли — по полю alive.
    for e in c.get_map()["enemies"]:
        print("  на карте:", e["kind"], "в", (e["x"], e["y"]), "| жив:", e["alive"], "| hp:", f'{e["hp"]}/{e["max_hp"]}')

    # Одно действие: шаг в сторону (D-069: движение — move_dir, 8 направлений, включая диагонали).
    res = c.move_dir("east" if ch["x"] < 6 else "west", reason="разогрев — пробую сходить")
    print("cooldown:", res["cooldown"], "c | новая позиция:", (res["character"]["x"], res["character"]["y"]))
    c.wait_cooldown()  # observe → decide → act → ВОТ ЭТО ОЖИДАНИЕ

Бой решается **одним вызовом** `c.fight()` — враг должен стоять **на твоей клетке** (D-069: дотянуться «по соседству» больше нельзя — шаг на клетку врага и есть цена боя). Вызов проигрывает всю стычку и возвращает в `result`:

- `combat_log` — `outcome` (`"win"` или `"death"` — бой идёт до конца), список `rounds`, `xp_gained`, `loot`, `summary` (средний урон, сколько съела броня врага);
- `gear_wear` — износ оружия за бой: `{wear, durability_after, active_next}` (или `null`, если дрался голыми руками). Копьё теряет **−1 прочности за бой**, а за гибель — ещё **−25% от максимума**;
- `warnings` — сервер сам предупредит, когда прочность на исходе;
- при гибели — `death_report`: **рюкзак сгорает целиком**, житель дома с 1 hp, но **экипировка не выпадает** — только изнашивается. Дальше жителя лечит **пассивный реген** — на домашней клетке он капает втрое быстрее (разбор — в секции 3).

Управлять hp между ударами внутри боя нельзя — решать «драться или нет» нужно **до** вызова. И для этого теперь есть инструмент честнее интуиции: **`c.fight_preview()`** — dry-run следующего боя на твоей клетке. Это **чистое чтение**: ни кулдауна, ни износа, ни риска, а раунды в прогнозе **в точности те**, что выдаст следующий реальный `fight()` (пока не изменились твои hp/статы/гир или враг). Бонусом — `projected_durability` (прочность после этого боя) и `on_death` (что именно сгорит, если погибнешь). Главный навык M2: **перед необратимым действием — сделай dry-run**.

In [ ]:
# Хелпер навигации — пригодится и ниже, в «Задачах»: один шаг к цели через move_dir (D-069).
DIR_BY_DELTA = {(0, -1): "north", (0, 1): "south", (1, 0): "east", (-1, 0): "west",
                (1, -1): "northeast", (-1, -1): "northwest", (1, 1): "southeast", (-1, 1): "southwest"}

def dir_toward(ch, tx, ty):
    """Направление одного шага к цели (север = y−1; диагонали срезают путь)."""
    dx = (tx > ch["x"]) - (tx < ch["x"])
    dy = (ty > ch["y"]) - (ty < ch["y"])
    return DIR_BY_DELTA.get((dx, dy))

if WORLD_UP:
    # Прогноз в деле — на БЕЗОПАСНОЙ цели: дойдём до ближайшего живого safe-врага (гоблина),
    # спросим прогноз и сравним с реальным боем. С опасными целями так и работают — см. «Задачи».
    SAFE_KINDS = {e["kind"] for e in c.get_enemies()["enemies"] if e["danger_at_base"] == "safe"}

    ch = c.get_character()
    safe = [e for e in c.get_map()["enemies"] if e["alive"] and e["kind"] in SAFE_KINDS]
    if not safe:
        print("safe-врагов сейчас нет (респаун ~15 с) — пропускаю демо; перезапусти ячейку чуть позже")
    else:
        goal = min(safe, key=lambda e: max(abs(e["x"] - ch["x"]), abs(e["y"] - ch["y"])))
        for _ in range(12):                       # бить можно только СТОЯ на клетке врага (D-069)
            ch = c.get_character()
            if (ch["x"], ch["y"]) == (goal["x"], goal["y"]):
                break
            c.move_dir(dir_toward(ch, goal["x"], goal["y"]), reason=f"иду к {goal['kind']} за прогнозом")
            c.wait_cooldown()
        try:
            preview = c.fight_preview()["result"]  # чистое чтение: ни кулдауна, ни риска
            plog = preview["combat_log"]
            print(f'прогноз: {plog["outcome"]} за {plog["summary"]["rounds"]} р. | мой hp после: {plog["player_hp"]}'
                  f' | прочность после: {preview["projected_durability"]}'
                  f' | если погибну, сгорит: {preview["on_death"]["backpack_lost"]}')
            if plog["outcome"] != "win":
                print("прогноз — смерть: НЕ бью (в этом и смысл dry-run). Подожди реген (hp капает сам, дома ×3) — или полечись мгновенно: c.heal_at_temple() — и перезапусти ячейку.")
            else:
                flog = c.fight(reason="бью по прогнозу — safe-цель")["result"]["combat_log"]
                print(f'факт:    {flog["outcome"]} за {flog["summary"]["rounds"]} р. | мой hp после: {flog["player_hp"]} | лут {flog["loot"]}')
                print("прогноз совпал с фактом раунд-в-раунд:", plog["rounds"] == flog["rounds"])
                c.wait_cooldown()
        except GameError as e:
            print("не вышло:", e.code, "— враг умер/ушёл в респаун; перезапусти ячейку")

## 3. Разбор — паттерн реактивного+ агента

Тот же цикл, что в M1, но `decide` теперь — **лесенка правил** (сверху вниз, срабатывает первое подходящее):

```
observe (character + map + бестиарий)  →  decide (лесенка)  →  act (ОДНО действие)  →  wait
```

1. **хозяйство** (base-действия, работают из любой клетки): трофеи на складе → `sell_trophy`; накопил на копьё → `buy_from_shop`; копьё куплено, но не надето → `equip`;
2. **логистика**: трофеи в рюкзаке → шаг к дому (на входе в `(0,0)` рюкзак сам банкуется на склад — а склад, в отличие от рюкзака, при смерти не горит). Тонкость: авто-банк кладёт только сколько влезет, поэтому каркас идёт домой лишь когда на складе есть место — забитый ещё с M1 склад лечится продажей трофеев или `upgrade("storehouse")`;
3. **выбор цели по бестиарию**: `safe` — всегда можно; `gear_required` — только с активным копьём; `wall` — **никогда** (огра агент обязан обходить);
4. **dry-run перед необратимым**: стоя на `gear_required`-цели — сперва `fight_preview()`; прогноз не `win` → не бить: ждать реген, пока не окрепнешь;
5. иначе: на клетке цели → `fight`, цель далеко → шаг к ней, целей нет (респаун) → ждать: hp тем временем капает сам.

Лечения-действия в M2 нет: hp восстанавливает **пассивный реген** на тике сервера — в поле 1 hp каждые 9 с, а на домашней клетке `(0,0)` — **втрое быстрее** (1 hp каждые 3 с; стат Выносливость ускоряет). Мгновенная альтернатива — **Храм**: `heal_at_temple()` лечит до полного откуда угодно, но за золото — 1 г за каждый недостающий hp. Это развилка «**время против золота**»: бесплатное лечение стоит времени (и дороги домой), мгновенное — денег, которые ты копишь на копьё. Свою скорость агент не угадывает, а **читает** из `get_character()["regen"]`: `hp_per_event`, периоды, `in_village` и готовый `seconds_to_full_here`.

Экономика кампании, которую агент проживает сам: ухо гоблина — 3 г нетто (Торговец сжигает 20% комиссии), копьё в Лавке — 30 г, то есть **~10 побед над гоблином**. Копьё даёт +2/+3 к атаке — и волк из «смерти с вероятностью 88%» превращается в надёжную добычу (шкура — 12 г нетто). Копьё можно и **скрафтить** (топорище + 3 камня на кузнице) — это путь урока M3 про планировщик; здесь идём через Лавку.

Ключевая мысль M2: **смотри перед тем, как бить** — вердикт бестиария вместо догадок и dry-run вместо храбрости. Детальный разбор боевой математики (броня, сид боя, износ) — в лекции урока.

## 4. Задачи — доведи бойца до ума

Ниже — **рабочий каркас**: справочники, хелперы, `decide()` и петля `run()`. Базовая версия уже ведёт полную кампанию «гоблины → золото → копьё → волк» и безопасна по построению: стартует от дома, делает всего `run(rounds=16)` ходов и **не бьёт опасную цель без прогноза**. Но до настоящего бойца ей не хватает двух привычек:

1. **TODO 1 — hp-guard.** Базовый агент лезет к гоблину с любым hp. Пока цель одна и safe — сходит с рук, но на длинном прогоне накопленный урон стачивает и здоровяка. Добавь правило: hp ниже `HP_GUARD_THRESHOLD` → сначала вылечись, потом бой. Лечение — развилка «время против золота»: бесплатно — отступить домой (`("step", HOME, ...)`, на `(0,0)` реген ×3) и переждать (`("wait", None, ...)`); мгновенно — Храм (`("heal", None, ...)`, 1 г за недостающий hp) — только не проешь накопления на копьё.
2. **TODO 2 — прочность и ремонт.** Копьё изнашивается: −1 за каждый бой и ещё −25% от максимума за гибель. На нуле оно **не выпадает из слота, но перестаёт работать** (`active=false`) — и волк снова смертелен. Изношенное копьё нельзя ни снять, ни заменить (`gear_worn`) — только чинить: `("repair", "weapon", ...)`, когда `needs_repair(ch)`. Ремонт — base-действие у **кузницы ≥ 1** (сырьё со склада: ~1 дерево + 1 камень за лёгкий износ). Кузницы нет? Построй: `c.upgrade("town_hall")` до L2 (15 дерева + 10 камня), затем `c.build("forge")` (8 дерева + 12 камня) — дерево и камень собираются как в M1.

Подсказки помечены `# TODO`. Ноутбук исполняется и до, и после правок — улучшай постепенно, а драму «что бывает без ремонта» оставь лекции.

In [ ]:
import time

# ---- справочники мира: агент не хардкодит цены и вердикты — он их ЧИТАЕТ из каталогов ----
BESTIARY = {e["kind"]: e for e in c.get_enemies()["enemies"]} if WORLD_UP else {}
VERDICT = {k: e["danger_at_base"] for k, e in BESTIARY.items()}          # safe | gear_required | wall
TROPHIES = {t["item"] for t in c.get_bounty()["trophies"]} if WORLD_UP else set()
SHOP = {i["item"]: i["unit_price"] for i in c.get_shop()["items"]} if WORLD_UP else {}
SPEAR, SPEAR_PRICE = "spear", SHOP.get("spear", 30)

HOME = (0, 0)

def steps_between(ax, ay, bx, by):
    """Дистанция в ходах при 8-направленном движении (метрика Чебышёва, диагональ = 1 ход)."""
    return max(abs(ax - bx), abs(ay - by))

def nearest_living(ch, enemies):
    """Ближайший ЖИВОЙ враг из списка. Живой — по полю e["alive"], не по картинке клетки."""
    alive = [e for e in enemies if e["alive"]]
    return min(alive, key=lambda e: steps_between(ch["x"], ch["y"], e["x"], e["y"])) if alive else None

def weapon(ch):
    return ch["equipment"]["weapon"]      # None или {item, durability, durability_max, active}

def spear_active(ch):
    w = weapon(ch)
    return w is not None and w["active"]  # active = прочность > 0; сломанное копьё в руке, но инертно

def owned_spear(ch):
    """Копьё в рюкзаке или на складе (ещё не надето)."""
    return ch["inventory"].get(SPEAR, 0) + ch["stored"].get(SPEAR, 0) > 0

def carried_trophies(ch):
    return {k: v for k, v in ch["inventory"].items() if k in TROPHIES and v > 0}

def stored_trophies(ch):
    return {k: v for k, v in ch["stored"].items() if k in TROPHIES and v > 0}

def stored_room(ch):
    """Свободное место на складе. Авто-банк кладёт только сколько влезет — на забитый склад
    (например, деревом с M1) трофей не ляжет, и бегать домой бессмысленно."""
    return ch["stored_cap"] - sum(ch["stored"].values())

def allowed_danger(ch):
    """Лесенка бестиария: safe — всегда; gear_required — только с активным оружием; wall — НИКОГДА."""
    return {"safe", "gear_required"} if spear_active(ch) else {"safe"}

def go_home(max_steps=14):
    """Тренировочные колёсики: вернуться домой (0, 0), чтобы демо было предсказуемым и безопасным."""
    for _ in range(max_steps):
        ch = c.get_character()
        if (ch["x"], ch["y"]) == HOME:
            return
        c.move_dir(dir_toward(ch, *HOME), reason="возвращаюсь домой перед вылазкой")
        c.wait_cooldown()

def regen_here(ch):
    """Реген на текущей клетке: (+hp за событие, период в секундах). Скорость агент ЧИТАЕТ из API
    (ch["regen"]), а не хардкодит: дома (0,0) реген втрое быстрее поля."""
    rg = ch["regen"]
    period = rg["village_period_s"] if rg["in_village"] else rg["field_period_s"]
    return rg["hp_per_event"], period

# Готовые инструменты для «Задач» (базовый decide() их пока НЕ использует):
HP_GUARD_THRESHOLD = 10  # ниже этого hp в бой не выходим: ждём реген (дома ×3) или платим Храму

def heal_cost(ch):
    """Цена мгновенного лечения в Храме (heal_at_temple): 1 золото за каждый недостающий hp."""
    return ch["max_hp"] - ch["hp"]

def needs_repair(ch):
    """Прочность просела ниже 25% максимума — тот же порог, с которого сервер шлёт warnings."""
    w = weapon(ch)
    return w is not None and w["durability"] * 4 < w["durability_max"]

In [ ]:
def decide(ch, world):
    """Верни (action, arg, reason). Одно решение — одно действие; правила читаются сверху вниз.

    БАЗОВАЯ версия уже ведёт кампанию «гоблины → золото → копьё → волк», но лечится только
    «в лоб» (ждёт реген там, где стоит) и не чинит копьё. Доработай по «Задачам».
    """
    # TODO 1 — hp-guard: не выходи в бой потрёпанным. Лечение — «время против золота»:
    #   if ch["hp"] < HP_GUARD_THRESHOLD:
    #       if ch["gold"] - heal_cost(ch) >= SPEAR_PRICE:    # золота с запасом → Храм: мгновенно, 1 г/hp
    #           return "heal", None, "плачу Храму за скорость — золото есть"
    #       if (ch["x"], ch["y"]) != HOME:                   # иначе платим временем: дома реген ×3
    #           return "step", HOME, "мало hp — отступаю домой под ×3-реген"
    #       return "wait", None, "дома: жду реген до безопасного hp"

    # TODO 2 — прочность: изношенное копьё чини, не жди поломки (нужна кузница ≥ 1 на базе):
    #   if needs_repair(ch):
    #       return "repair", "weapon", "копьё на исходе — чиню в кузнице"

    # 1) хозяйство — base-действия, работают из любой клетки:
    if stored_trophies(ch):
        kind = next(iter(stored_trophies(ch)))
        return "sell", kind, f"продаю {kind} со склада — коплю на копьё"
    if not spear_active(ch) and not owned_spear(ch) and ch["gold"] >= SPEAR_PRICE and stored_room(ch) > 0:
        # покупка приезжает на СКЛАД — в забитый склад Лавка не продаст (storehouse_full);
        # лечится продажей трофеев (rule выше) или upgrade("storehouse")
        return "buy", SPEAR, f"в казне {ch['gold']} г — покупаю копьё в Лавке"
    if not spear_active(ch) and owned_spear(ch):
        return "equip", SPEAR, "надеваю копьё — теперь волк по зубам"

    # 2) логистика: трофеи в рюкзаке → домой (на входе в (0,0) рюкзак сам банкуется на склад).
    #    Но только если склад ПРИМЕТ трофей — иначе агент бегал бы домой впустую.
    if carried_trophies(ch) and stored_room(ch) > 0 and (ch["x"], ch["y"]) != HOME:
        return "step", HOME, "несу трофеи домой на склад"

    # 3) цель по лесенке бестиария: safe всегда, gear_required — только с копьём, wall — никогда
    hunt = [e for e in world["enemies"] if VERDICT.get(e["kind"]) in allowed_danger(ch)]
    enemy = nearest_living(ch, hunt)
    if enemy is None:
        return "wait", None, "посильных живых целей нет (респаун) — жду; hp тем временем капает сам"
    if (ch["x"], ch["y"]) != (enemy["x"], enemy["y"]):
        return "step", (enemy["x"], enemy["y"]), f'иду к {enemy["kind"]} ({VERDICT[enemy["kind"]]})'

    # 4) стоим на цели. Перед НЕОБРАТИМЫМ действием — dry-run:
    if VERDICT[enemy["kind"]] == "gear_required":
        plog = c.fight_preview()["result"]["combat_log"]   # чистое чтение — без кулдауна и риска
        if plog["outcome"] != "win":
            return "wait", None, "прогноз — смерть; жду реген, пока не окрепну"
    return "fight", None, f'бью {enemy["kind"]}'


DEATHS = 0  # гибели за прогон: «Проверка» ниже требует ноль

def run(rounds=16):
    global DEATHS
    go_home()  # тренировочные колёсики: старт от дома делает короткий прогон предсказуемым
    for _ in range(rounds):
        ch = c.get_character()                       # observe: себя (hp, золото, снаряжение)
        world = c.get_map()                          # observe: мир (кто жив и где)
        try:
            action, arg, reason = decide(ch, world)  # decide
            if action == "wait":                     # act — ровно одно действие за ход:
                per_event, period = regen_here(ch)   # лечение пассивно — пережидаем один тик регена
                print(f'  жду: +{per_event} hp / {period:.0f} с ({"деревня ×3" if ch["regen"]["in_village"] else "поле"})'
                      f' | hp {ch["hp"]}/{ch["max_hp"]}, до полного ~{ch["regen"]["seconds_to_full_here"]:.0f} с')
                time.sleep(period + 1)               # +1 с поглощает фазу тика
            elif action == "heal":
                r = c.heal_at_temple(reason=reason)["result"]
                print(f'  Храм: +{r["healed"]} hp за {r["cost"]} г | hp {r["hp"]} | в казне {r["gold"]} г')
            elif action == "sell":
                r = c.sell_trophy(arg, qty=ch["stored"][arg], reason=reason)["result"]
                print(f'  продал {r["qty"]}×{r["sold"]}: +{r["net"]} г (комиссия {r["burned"]}) | в казне {r["gold"]} г')
            elif action == "buy":
                r = c.buy_from_shop(arg, reason=reason)["result"]
                print(f'  куплено: {r["bought"]} за {r["cost"]} г | в казне {r["gold"]} г')
            elif action == "equip":
                r = c.equip(arg, reason=reason)["result"]
                print(f'  надето: {r["equipped"]} | прочность {r["durability"]}/{r["durability_max"]}')
            elif action == "repair":
                r = c.repair(arg, reason=reason)["result"]
                print(f'  починка: прочность {r["durability"]} (+{r["restored"]}) за {r["cost"]}')
            elif action == "step":
                c.move_dir(dir_toward(ch, *arg), reason=reason)
            elif action == "fight":
                r = c.fight(reason=reason)["result"]
                log, wear = r["combat_log"], r["gear_wear"]
                print(f'  бой с {log["enemy"]}: {log["outcome"]} за {log["summary"]["rounds"]} р.'
                      f' | xp+{log["xp_gained"]} | лут {log["loot"]}'
                      + (f' | копьё: {wear["durability_after"]} прочн. (−{wear["wear"]})' if wear else ""))
                if log["outcome"] == "death":
                    DEATHS += 1              # смерть: рюкзак сгорел, копьё −25% прочности, дом с 1 hp
                for warn in r.get("warnings") or []:
                    print("  ⚠️", warn)
        except GameError as e:
            print("  не вышло:", e.code)  # напр. no_enemy_here — враг умер раньше вас (мир общий)
            time.sleep(1)                 # пауза: не жечь rate-limit, пока причина не рассосётся
        c.wait_cooldown()                 # wait — честный ритм общего мира

if WORLD_UP:
    run()

## 5. Проверка

Главный критерий M2 — **житель не погибал за прогон** (смерть наказывает: рюкзак сгорает, копьё теряет −25% прочности, дом встречает с 1 hp); базовый агент это проходит. Полная кампания в короткий `Run all` не помещается по построению: ~10 побед над гоблином — это ~150–200 ходов с респауном. Прогресс не сгорает между прогонами (золото — в казне, копьё — в слоте, склад цел), так что подними `rounds` в `run()` или просто перезапускай ячейку — ячейка ниже покажет, на каком этапе кампании ты сейчас.

In [ ]:
if WORLD_UP:
    ch = c.get_character()
    w = weapon(ch)
    per_event, period = regen_here(ch)
    print("hp:", f'{ch["hp"]}/{ch["max_hp"]}',
          "| реген здесь:", f'+{per_event} hp / {period:.0f} с' + (" (деревня ×3)" if ch["regen"]["in_village"] else " (поле)"),
          "| золото:", ch["gold"], "| xp:", ch["xp"],
          "| оружие:", f'{w["item"]} {w["durability"]}/{w["durability_max"]} (active={w["active"]})' if w else "нет")
    assert ch["hp"] > 0 and DEATHS == 0, \
        "Житель погибал в прогоне: рюкзак сгорел, копьё −25% прочности. В этом и урок M2: dry-run перед боем и hp-guard (см. Задачи)."

    wolf_win = any(e["action"] == "fight" and "победа над wolf" in e["summary"]
                   for e in c.get_events(limit=200))
    if wolf_win:
        print("✅ кампания пройдена: волк повержен с копьём — полная петля M2 закрыта")
    elif spear_active(ch):
        print("✅ копьё в руке — осталось добыть волка: подними rounds в run() и доверяй прогнозу")
    elif ch["gold"] > 0 or stored_trophies(ch) or carried_trophies(ch):
        print(f"✅ выжил и копит: {ch['gold']} г из {SPEAR_PRICE} г на копьё — продолжай фарм (подними rounds)")
    else:
        print("✅ выжил. Побед пока не видно (гоблин мог быть на респауне) — перезапусти ячейку run()")
else:
    print("Мир недоступен — проверка пропущена.")

## Наблюдаемость — смотри за умом своего бойца

Открой в браузере `BASE_URL/?token=<TOKEN>` (ссылка напечатана в разогреве): в одной вкладке крутится агент, в другой видно его шаги, **мысль-пузырь** (`reason`, который ты передаёшь в действия) и Хронику боёв — включая строки про износ и починку копья. Клик по имени жителя в ростере открывает **паспорт**: боевой лист и слот оружия с прочностью. Это и есть «1 житель = 1 агент»: ты пишешь правила безопасности, а наблюдаешь живого бойца.

Ручной тык по API — `BASE_URL/docs` (кнопка **Authorize**, токен один раз без префикса `Bearer`).